# FINAL: Alzheimer's Phase 1 → Phase 3 Prediction Model

## Enhanced Feature Set - NO DATA LEAKAGE

This notebook uses **comprehensive trial design features** to predict Phase 3 success.

### Feature Categories (Total: ~80+ features):

#### 1. **Basic Trial Characteristics** (15 features)
- Enrollment size, number of sites, countries
- Trial duration and follow-up length
- Number of arms and interventions

#### 2. **Outcome Measures Design** (10 features)
- ✅ Has cognitive endpoints (ADAS-Cog, MMSE, CDR)
- ✅ Has biomarker endpoints (amyloid PET, CSF tau)
- ✅ Has functional endpoints (ADL, IADL)
- ✅ Long-term follow-up (>6 months)
- ✅ Number and complexity of outcome measures

#### 3. **Study Design Sophistication** (15 features)
- ✅ Randomization and allocation concealment
- ✅ Blinding level (double, triple, quadruple)
- ✅ Who is masked (participant, investigator, assessor)
- ✅ Study model (parallel, crossover, factorial)
- ✅ Placebo control presence
- ✅ Dose escalation design

#### 4. **Patient Population Targeting** (10 features)
- ✅ Targets MCI vs mild AD vs moderate AD
- ✅ Biomarker requirements (PET, CSF)
- ✅ Genetic testing requirements (APOE)
- ✅ Age range and inclusivity
- ✅ Eligibility criteria complexity

#### 5. **Geographic & Site Quality** (8 features)
- ✅ Number of countries and US states
- ✅ Regional presence (US, Europe, Asia)
- ✅ Multi-site vs single-site
- ✅ Large network (>20 sites)

#### 6. **Sponsor & Funding** (8 features)
- ✅ Sponsor type (Industry, NIH, Academic)
- ✅ Industry collaboration
- ✅ Number of collaborators
- ✅ Funding diversity

#### 7. **Regulatory & Oversight** (5 features)
- ✅ Data Monitoring Committee (DMC) presence
- ✅ FDA regulation status
- ✅ Section 801 compliance

#### 8. **Prior Research Context** (5 features)
- ✅ Number of references cited
- ✅ Has prior research foundation
- ✅ Keyword richness

#### 9. **Phase 2 Design** (3 features - NO LEAKAGE)
- ✅ Phase 2 was planned from start
- ✅ Combined Phase 1/2 design
- ✅ Early Phase 1 designation

### ❌ **What We DON'T Use (Would Cause Leakage)**:
- Actual efficacy results from Phase 1/2
- Adverse event rates from Phase 1/2
- Actual enrollment vs planned
- Protocol amendments
- Interim analysis results

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Try to import XGBoost, but continue if it fails
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("✓ XGBoost loaded")
except Exception as e:
    XGBOOST_AVAILABLE = False
    print("⚠️ XGBoost not available (this is OK, we'll use other models)")
    print(f"   Error: {str(e)[:100]}")
    print("   To fix: Run 'brew install libomp' in terminal, then restart kernel")

from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve,
    precision_recall_curve, average_precision_score,
    make_scorer
)
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Core libraries loaded")

✓ XGBoost loaded
✓ Core libraries loaded


## 1. Load Enhanced Features

In [4]:
# Try to load enhanced features, fall back to basic if not available
try:
    df = pd.read_csv('/Users/shreyamendi/clinical-trial-success-predictor/data/alzheimers_final_enhanced.csv', low_memory=False)
    print("✓ Loaded enhanced features dataset")
except:
    print("⚠️ Enhanced features not found. Run enhanced_feature_engineering.ipynb first!")
    print("Loading basic features...")
    df = pd.read_csv('/Users/shreyamendi/clinical-trial-success-predictor/data/all_trials_parsed.csv', low_memory=False)
    df = df[df['is_alzheimers'] == 1].copy()

print(f"\nDataset: {len(df)} trials, {len(df.columns)} features")

✓ Loaded enhanced features dataset

Dataset: 5187 trials, 88 features


## 2. Create Target Variable

In [5]:
def classify_trial_outcome(row):
    """Classify based on maximum phase reached."""
    phase_str = str(row['phase']).lower()
    status = str(row['overall_status']).lower()
    
    if 'phase 4' in phase_str:
        return 4
    elif 'phase 3' in phase_str or 'phase 2/phase 3' in phase_str:
        return 3
    elif 'phase 2' in phase_str or 'phase 1/phase 2' in phase_str:
        return 2
    elif 'phase 1' in phase_str or 'early phase 1' in phase_str:
        if any(term in status for term in ['terminated', 'withdrawn', 'suspended', 'completed']):
            return 1
    return None

if 'max_phase_reached' not in df.columns:
    df['max_phase_reached'] = df.apply(classify_trial_outcome, axis=1)

df['reached_phase2'] = (df['max_phase_reached'] >= 2).astype(int)
df['reached_phase3'] = (df['max_phase_reached'] >= 3).astype(int)

# Filter for trials with clear outcomes
df_model = df[df['max_phase_reached'].notna()].copy()

print(f"Trials with clear outcomes: {len(df_model)}")
print(f"\nPhase distribution:")
print(df_model['max_phase_reached'].value_counts().sort_index())
print(f"\nSuccess rates:")
print(f"  Phase 1 → Phase 2: {df_model['reached_phase2'].mean():.2%}")
print(f"  Phase 1 → Phase 3: {df_model['reached_phase3'].mean():.2%}")

Trials with clear outcomes: 1773

Phase distribution:
max_phase_reached
1.0    435
2.0    749
3.0    406
4.0    183
Name: count, dtype: int64

Success rates:
  Phase 1 → Phase 2: 75.47%
  Phase 1 → Phase 3: 33.22%


## 3. Feature Engineering & Selection

In [6]:
# Define comprehensive feature list
feature_cols = []

# Basic trial characteristics
basic_features = [
    'enrollment', 'num_collaborators', 'num_conditions', 'num_interventions',
    'num_locations', 'num_countries', 'num_arm_groups', 'num_keywords',
    'criteria_length', 'num_primary_outcomes', 'num_secondary_outcomes'
]

# Binary design features
binary_features = [
    'has_collaborators', 'has_drug_intervention', 'has_biological_intervention',
    'is_international', 'has_dmc',
    'is_randomized', 'is_double_blind', 'is_parallel', 'is_treatment',
    'accepts_healthy', 'gender_all',
    'is_fda_regulated_drug', 'is_fda_regulated_device'
]

# Phase 2 design (no leakage)
phase2_design = [
    'phase2_planned', 'is_combined_phase1_2', 'is_early_phase1'
]

# Enhanced outcome measures
outcome_features = [
    'has_cognitive_endpoint', 'has_biomarker_endpoint', 'has_functional_endpoint',
    'max_followup_weeks', 'has_longterm_followup', 'total_outcome_measures',
    'has_many_outcomes', 'primary_outcome_text_length'
]

# Enhanced design sophistication
design_features = [
    'has_placebo', 'has_dose_escalation', 'has_multiple_doses',
    'is_triple_blind', 'is_quadruple_blind',
    'masks_participant', 'masks_investigator', 'masks_outcomes_assessor',
    'is_crossover', 'is_factorial', 'is_sequential'
]

# Patient population
population_features = [
    'num_inclusion_criteria', 'num_exclusion_criteria',
    'requires_biomarker', 'requires_genetic_test',
    'targets_mci', 'targets_mild_ad', 'targets_moderate_ad',
    'min_age_years', 'max_age_years', 'age_range'
]

# Geographic features
geo_features = [
    'num_us_states', 'is_us_only', 'includes_us',
    'includes_europe', 'includes_asia',
    'is_multisite', 'is_large_network'
]

# Sponsor features
sponsor_features = [
    'sponsor_is_industry', 'sponsor_is_nih', 'sponsor_is_academic',
    'has_industry_collab', 'has_nih_collab'
]

# Research context
research_features = [
    'num_references', 'has_prior_research'
]

# Combine all
all_feature_groups = [
    basic_features, binary_features, phase2_design,
    outcome_features, design_features, population_features,
    geo_features, sponsor_features, research_features
]

for group in all_feature_groups:
    feature_cols.extend([f for f in group if f in df_model.columns])

# Remove duplicates
feature_cols = list(set(feature_cols))

print(f"Total features available: {len(feature_cols)}")
print(f"\nFeature breakdown:")
print(f"  Basic: {sum(f in df_model.columns for f in basic_features)}")
print(f"  Binary design: {sum(f in df_model.columns for f in binary_features)}")
print(f"  Phase 2 design: {sum(f in df_model.columns for f in phase2_design)}")
print(f"  Outcome measures: {sum(f in df_model.columns for f in outcome_features)}")
print(f"  Design sophistication: {sum(f in df_model.columns for f in design_features)}")
print(f"  Population: {sum(f in df_model.columns for f in population_features)}")
print(f"  Geographic: {sum(f in df_model.columns for f in geo_features)}")
print(f"  Sponsor: {sum(f in df_model.columns for f in sponsor_features)}")
print(f"  Research: {sum(f in df_model.columns for f in research_features)}")

Total features available: 61

Feature breakdown:
  Basic: 11
  Binary design: 7
  Phase 2 design: 3
  Outcome measures: 8
  Design sophistication: 11
  Population: 7
  Geographic: 7
  Sponsor: 5
  Research: 2


In [7]:
# Fill missing values
for col in feature_cols:
    df_model[col] = df_model[col].fillna(0)

# Create log transforms for skewed features
for col in ['enrollment', 'criteria_length', 'num_locations']:
    if col in feature_cols:
        log_col = f'log_{col}'
        df_model[log_col] = np.log1p(df_model[col])
        if log_col not in feature_cols:
            feature_cols.append(log_col)

print(f"\nFinal feature count: {len(feature_cols)}")


Final feature count: 64


## 4. Prepare Data for Modeling

In [8]:
# Choose target: predicting Phase 3 success
TARGET = 'reached_phase3'

X = df_model[feature_cols].copy()
y = df_model[TARGET].copy()

print(f"Dataset: {len(X)} samples, {len(feature_cols)} features")
print(f"Target: {TARGET}")
print(f"  Positive: {y.sum()} ({y.mean():.2%})")
print(f"  Negative: {(~y.astype(bool)).sum()} ({(1-y.mean()):.2%})")
print(f"  Imbalance ratio: {(~y.astype(bool)).sum() / max(y.sum(), 1):.1f}:1")

# Check if we have enough positive samples
if y.sum() < 10:
    print("\n⚠️ WARNING: Very few positive samples. Model may not be reliable.")
    print("Consider: 1) Expanding to Phase 2 as success, or 2) Collecting more data")
    CAN_MODEL = False
else:
    CAN_MODEL = True
    print("\n✓ Sufficient data for modeling")

Dataset: 1773 samples, 64 features
Target: reached_phase3
  Positive: 589 (33.22%)
  Negative: 1184 (66.78%)
  Imbalance ratio: 2.0:1

✓ Sufficient data for modeling


In [9]:
if CAN_MODEL:
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Training: {len(X_train)} samples ({y_train.sum()} positive)")
    print(f"Test: {len(X_test)} samples ({y_test.sum()} positive)")
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Handle imbalance with SMOTE
    smote = SMOTE(random_state=42, k_neighbors=min(5, y_train.sum()-1))
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
    
    print(f"\nAfter SMOTE: {len(X_train_balanced)} samples")
    print(pd.Series(y_train_balanced).value_counts())

Training: 1418 samples (471 positive)
Test: 355 samples (118 positive)

After SMOTE: 1894 samples
reached_phase3
1    947
0    947
Name: count, dtype: int64


## 5. Model Training - Multiple Algorithms

In [ ]:
if CAN_MODEL:
    print("Training multiple models...\n")
    
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, C=0.1),
        'Random Forest': RandomForestClassifier(random_state=42, n_estimators=200, max_depth=10, min_samples_split=10),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=200, learning_rate=0.05, max_depth=5),
        'Extra Trees': ExtraTreesClassifier(random_state=42, n_estimators=200, max_depth=10)
    }
    
    # Add XGBoost if available
    if XGBOOST_AVAILABLE:
        models['XGBoost'] = XGBClassifier(random_state=42, n_estimators=200, learning_rate=0.05, max_depth=5, eval_metric='logloss')
        print("Including XGBoost in model comparison")
    else:
        print("Skipping XGBoost (not available)")
    
    print(f"\nTraining {len(models)} models...\n")
    
    results = {}
    trained_models = {}
    
    for name, model in models.items():
        print(f"Training {name}...")
        
        model.fit(X_train_balanced, y_train_balanced)
        trained_models[name] = model
        
        # Predictions
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        
        # Metrics
        results[name] = {
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1': f1_score(y_test, y_pred, zero_division=0),
            'ROC-AUC': roc_auc_score(y_test, y_pred_proba),
            'PR-AUC': average_precision_score(y_test, y_pred_proba)
        }
        
        print(f"  ROC-AUC: {results[name]['ROC-AUC']:.4f}")
        print(f"  Precision: {results[name]['Precision']:.4f}")
        print(f"  Recall: {results[name]['Recall']:.4f}\n")
    
    # Compare models
    results_df = pd.DataFrame(results).T
    results_df = results_df.round(4)
    
    print("\n" + "="*80)
    print("MODEL COMPARISON")
    print("="*80)
    print(results_df.sort_values('ROC-AUC', ascending=False))
    
    # Select best model
    best_model_name = results_df['ROC-AUC'].idxmax()
    best_model = trained_models[best_model_name]
    
    print(f"\n🏆 Best Model: {best_model_name}")
    print(f"   ROC-AUC: {results_df.loc[best_model_name, 'ROC-AUC']:.4f}")

## 6. Model Evaluation & Visualizations

In [ ]:
if CAN_MODEL:
    # Get predictions from best model
    y_pred = best_model.predict(X_test_scaled)
    y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Confusion matrix heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
    axes[0].set_title(f'Confusion Matrix - {best_model_name}')
    axes[0].set_ylabel('True')
    axes[0].set_xlabel('Predicted')
    axes[0].set_xticklabels(['Failed', 'Success'])
    axes[0].set_yticklabels(['Failed', 'Success'])
    
    # Confusion matrix metrics
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    
    metrics_text = f"""Metrics:
Sensitivity (Recall): {sensitivity:.3f}
Specificity: {specificity:.3f}
PPV (Precision): {ppv:.3f}
NPV: {npv:.3f}

TP: {tp}, FP: {fp}
FN: {fn}, TN: {tn}"""
    
    axes[1].text(0.1, 0.5, metrics_text, fontsize=12, family='monospace',
                verticalalignment='center')
    axes[1].axis('off')
    axes[1].set_title('Performance Metrics')
    
    plt.tight_layout()
    plt.show()

In [ ]:
if CAN_MODEL:
    # ROC and PR curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    axes[0].plot(fpr, tpr, linewidth=2, label=f'ROC (AUC = {roc_auc:.3f})')
    axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Curve')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)
    baseline = y_test.mean()
    
    axes[1].plot(recall, precision, linewidth=2, label=f'PR (AUC = {pr_auc:.3f})')
    axes[1].axhline(baseline, color='k', linestyle='--', linewidth=1, label=f'Baseline ({baseline:.3f})')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall Curve')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
if CAN_MODEL:
    # Classification report
    print("\n" + "="*80)
    print("CLASSIFICATION REPORT")
    print("="*80)
    print(classification_report(y_test, y_pred, 
                                target_names=['Did Not Reach Phase 3', 'Reached Phase 3'],
                                zero_division=0))

## 7. Feature Importance Analysis

In [ ]:
if CAN_MODEL and hasattr(best_model, 'feature_importances_'):
    # Get feature importances
    importances = best_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    # Plot top 25 features
    top_n = min(25, len(feature_importance_df))
    top_features = feature_importance_df.head(top_n)
    
    plt.figure(figsize=(10, 12))
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Feature Importance')
    plt.title(f'Top {top_n} Most Important Features - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("\nTop 20 Features:")
    print(feature_importance_df.head(20).to_string(index=False))

## 8. Final Summary & Model Saving

In [ ]:
print("\n" + "="*80)
print("FINAL MODEL SUMMARY")
print("="*80)

print("\n1. DATASET")
print("-" * 80)
print(f"Total Alzheimer's trials analyzed: {len(df)}")
print(f"Trials with clear outcomes: {len(df_model)}")
print(f"Success rate (Phase 3): {y.mean():.2%}")

if CAN_MODEL:
    print("\n2. FEATURES")
    print("-" * 80)
    print(f"Total features: {len(feature_cols)}")
    print("Feature categories:")
    print("  • Basic trial characteristics")
    print("  • Outcome measure design (cognitive, biomarker, functional)")
    print("  • Study design sophistication (blinding, randomization)")
    print("  • Patient population targeting (MCI, AD severity)")
    print("  • Geographic scope (multi-country, regions)")
    print("  • Sponsor and collaboration patterns")
    print("  • Regulatory oversight (DMC, FDA)")
    print("  • Phase 2 design planning (NO LEAKAGE)")
    
    print("\n3. MODEL PERFORMANCE")
    print("-" * 80)
    print(f"Best Model: {best_model_name}")
    print(f"\nTest Set Metrics:")
    print(f"  • ROC-AUC: {roc_auc:.4f}")
    print(f"  • PR-AUC: {pr_auc:.4f}")
    print(f"  • Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"  • Recall: {recall_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"  • F1-Score: {f1_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"  • Specificity: {specificity:.4f}")
    print(f"  • NPV: {npv:.4f}")

print("\n4. DATA LEAKAGE PREVENTION")
print("-" * 80)
print("✓ Only trial DESIGN features used (not results)")
print("✓ No efficacy data from Phase 1/2")
print("✓ No adverse event rates from Phase 1/2")
print("✓ No actual enrollment data (only planned)")
print("✓ Phase 2 design features are planning decisions, not outcomes")
print("✓ Model can be applied immediately after Phase 1 design is finalized")

print("\n" + "="*80)

In [ ]:
if CAN_MODEL:
    # Save model
    import pickle
    
    model_artifacts = {
        'model': best_model,
        'model_name': best_model_name,
        'scaler': scaler,
        'feature_cols': feature_cols,
        'feature_importance': feature_importance_df if hasattr(best_model, 'feature_importances_') else None,
        'performance_metrics': results_df.loc[best_model_name].to_dict(),
        'training_date': pd.Timestamp.now().strftime('%Y-%m-%d'),
        'n_features': len(feature_cols),
        'n_train': len(X_train),
        'n_test': len(X_test)
    }
    
    import os
    os.makedirs('/Users/shreyamendi/clinical-trial-success-predictor/models', exist_ok=True)
    
    model_path = '/Users/shreyamendi/clinical-trial-success-predictor/models/final_alzheimers_predictor.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump(model_artifacts, f)
    
    print(f"\n✅ Model saved to: {model_path}")
    print("\nTo use this model:")
    print("  1. Load the pickle file")
    print("  2. Prepare new trial data with the same features")
    print("  3. Scale using the saved scaler")
    print("  4. Get predictions with model.predict_proba()")